# Section III: LoRA Fine-Tuning Walkthrough

**Talk section**: Section III — The Proposed Framework

This notebook walks through the LoRA fine-tuning process for one trader persona (momentum). The goal is to show, step by step, how we take the synthetic reasoning traces generated in Notebook 01 and use them to create a persona-specific behavioral adapter for `Qwen/Qwen2.5-1.5B-Instruct`.

By the end of this notebook you will understand:
- What LoRA is and why it is appropriate for persona adaptation
- How we format the training data as a chat conversation
- The specific LoRA configuration choices we make and why
- How to train (requires GPU) or load a pre-trained adapter (CPU-friendly)
- How to run sample inference with the fine-tuned adapter

## Why LoRA?

**LoRA** (Low-Rank Adaptation, Hu et al. 2022) is a parameter-efficient fine-tuning technique. Instead of updating all the weights of a large model, LoRA freezes the original model and adds small, trainable low-rank matrices to the attention layers.

For our use case, LoRA is ideal for three reasons:

1. **Multiple personas, shared base**: We need three distinct behavioral adapters. With LoRA, all three share the same frozen base model — only the adapters (~2M parameters each) differ. At inference time, we can hot-swap adapters to instantiate agents of different types, without loading three separate 1.5B-parameter models.

2. **Small dataset regime**: We have only ~300 examples per persona. Fine-tuning all 1.5B parameters on 300 examples would massively overfit. LoRA's low-rank constraint acts as a strong regularizer — the adapter can only learn low-dimensional modifications to the model's behavior.

3. **Interpretability signal**: The adapter weights are small and concentrated. In principle, we can compare the adapter weights across personas to understand how they differ geometrically. This is a potential direction for the interpretability literature on persona-conditioned models.

**The math**: A LoRA adapter for weight matrix W (shape d×k) adds two low-rank matrices A (d×r) and B (r×k) where r << min(d,k). The adapted weight is W + AB. We freeze W and train A and B. With rank r=8 on attention projections, this adds roughly 2M trainable parameters to a 1.5B-parameter model — less than 0.15%.

In [ ]:
# Imports
import json
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")

# Check GPU availability
try:
    import torch
    gpu_available = torch.cuda.is_available()
    print(f"GPU available: {gpu_available}")
    if gpu_available:
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    else:
        print("No GPU detected. Training will be very slow on CPU.")
        print("This notebook will demonstrate the pipeline but skip the actual training step.")
        print("To train, use a machine with an A100 GPU (~15 min per persona).")
except ImportError:
    print("torch not installed. Install with: pip install torch")
    gpu_available = False

## Data Format: Chat Conversations

The training data (from Notebook 01) is stored as flat JSON objects:
```json
{"persona": "momentum", "market_state": {...}, "reasoning": "...", "action": "BUY", "quantity": 10}
```

For fine-tuning with SFTTrainer, we need to reformat this as a **chat conversation**:
- **System turn**: The persona identity string
- **User turn**: The market state description (same format used at simulation time)
- **Assistant turn**: The reasoning chain + action JSON (this is what we train the model to produce)

We train only on the **assistant turn** (the completion). The loss is masked on the system and user turns, so the model learns to produce the persona-appropriate reasoning, not to reproduce the prompt.

This is the standard approach for instruction fine-tuning. The key constraint is that the user prompt format must exactly match what will be used during simulation inference.

In [ ]:
# Load and display the momentum persona data

def load_jsonl(path):
    examples = []
    with open(path) as f:
        for line in f:
            if line.strip():
                examples.append(json.loads(line))
    return examples

momentum_data = load_jsonl(project_root / "data" / "personas" / "momentum.jsonl")
print(f"Loaded {len(momentum_data)} momentum examples.")
print()

# Show the raw format of the first example
print("Raw training example format:")
print(json.dumps(momentum_data[0], indent=2))

In [ ]:
# Show the chat-formatted version of the same example
# This is the format that SFTTrainer expects

PERSONA_IDENTITIES = {
    "momentum": (
        "You are a momentum trader. You believe recent price trends persist in the short run. "
        "You buy assets that have risen recently and sell assets that have fallen. "
        "You do not anchor to fundamentals."
    ),
}

def format_as_chat(example):
    """Convert a flat training example to a chat conversation dict."""
    persona = example["persona"]
    ms = example["market_state"]
    prices = ms["price_history"]
    current_price = prices[-1]
    price_change = round((current_price - prices[0]) / prices[0] * 100, 2)
    
    user_content = (
        f"You are analyzing a financial market. Here is the current market state:\n\n"
        f"Price history (last 5 periods): {prices}\n"
        f"Current price: {current_price}\n"
        f"Price change over window: {price_change}%\n"
        f"Latest news: {ms['news']}\n"
        f"Estimated fair value: {ms['fair_value']}\n\n"
        f"Based on your identity as a trader, reason step-by-step about what you observe "
        f"and what action you will take.\n\n"
        f"Provide your decision in this JSON format:\n"
        '{\n  "reasoning": "...",\n  "action": "BUY/SELL/HOLD",\n  "quantity": 0\n}'
    )
    
    assistant_content = json.dumps({
        "reasoning": example["reasoning"],
        "action": example["action"],
        "quantity": example["quantity"],
    }, indent=2)
    
    return {
        "messages": [
            {"role": "system", "content": PERSONA_IDENTITIES[persona]},
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": assistant_content},
        ]
    }

chat_example = format_as_chat(momentum_data[0])
print("Chat-formatted example:")
print(f"  SYSTEM: {chat_example['messages'][0]['content'][:80]}...")
print(f"  USER:   {chat_example['messages'][1]['content'][:80]}...")
print(f"  ASSISTANT: {chat_example['messages'][2]['content']}")

## LoRA Configuration

The key LoRA hyperparameters are:

- **r = 8**: The rank of the adapter matrices. Higher rank = more capacity but more parameters. Rank 8 is a standard default that works well for small datasets. With rank 8 on Q and V projections, we add ~2M trainable parameters to the 1.5B-parameter base model.

- **lora_alpha = 16**: The scaling factor, applied as `alpha/r`. With alpha=16 and r=8, the effective scaling is 2.0. This means the adapter's contribution is scaled up by 2x relative to the base weight, giving it more influence during training. The alpha/r ratio is more important than the absolute values.

- **target_modules = ["q_proj", "v_proj"]**: We adapt only the query and value projection matrices in the attention layers. The original LoRA paper showed these are the most impactful for task adaptation. We skip k_proj and the MLP layers to keep the adapter smaller.

- **lora_dropout = 0.05**: A small dropout rate on the adapter activations. This regularizes the adapter and prevents overfitting on our small dataset.

All three personas use identical LoRA configurations. The behavioral differences emerge entirely from the training data, not from any structural differences in the adapters.

In [ ]:
# Show the LoRA configuration for the momentum persona
# This is exactly what is read from training/configs/momentum.yaml

import yaml

config_path = project_root / "training" / "configs" / "momentum.yaml"
with open(config_path) as f:
    config = yaml.safe_load(f)

print("Momentum LoRA configuration:")
print()

print(f"  Base model: {config['base_model_name_or_path']}")
print()
print("  LoRA parameters:")
print(f"    rank (r):          {config['lora_r']}")
print(f"    alpha:             {config['lora_alpha']}")
print(f"    dropout:           {config['lora_dropout']}")
print(f"    target modules:    {config['target_modules']}")
print(f"    effective scaling: {config['lora_alpha'] / config['lora_r']:.1f}x")
print()
print("  Training parameters:")
print(f"    epochs:            {config['num_train_epochs']}")
print(f"    batch size:        {config['per_device_train_batch_size']}")
print(f"    learning rate:     {config['learning_rate']}")
print(f"    max seq length:    {config['max_seq_length']}")
print()
print(f"  Output dir: {config['output_dir']}")

# Compute approximate parameter counts
# For Qwen2.5-1.5B: hidden_dim = 1536, attention has 12 heads, head_dim = 128
hidden_dim = 1536
r = config['lora_r']
# q_proj: (hidden_dim x hidden_dim), v_proj: (hidden_dim x hidden_dim)
# LoRA adds A (hidden_dim x r) and B (r x hidden_dim) for each
n_layers = 28  # Qwen2.5-1.5B has 28 layers
params_per_module = 2 * (hidden_dim * r + r * hidden_dim)  # A and B matrices
total_lora_params = n_layers * 2 * params_per_module  # 2 modules per layer
print(f"\n  Estimated trainable parameters: ~{total_lora_params/1e6:.1f}M")
print(f"  Base model parameters:          ~1,500M")
print(f"  Trainable fraction:             ~{total_lora_params/1500e6*100:.2f}%")

## Training with SFTTrainer

We use **SFTTrainer** from the TRL (Transformer Reinforcement Learning) library. SFTTrainer handles:
- Formatting the chat template for the specific model (Qwen2.5 uses a custom template)
- Masking the loss on the prompt turns (we only train on the assistant's response)
- Gradient accumulation to simulate larger batch sizes
- Integration with PEFT adapters

**Training requirements**:
- GPU: A single A100 (40GB) is ideal. Training takes ~15 minutes per persona.
- Memory: The model in bfloat16 uses ~3GB. With gradients and optimizer state, you need at least 12GB VRAM.
- CPU training: Technically possible but would take many hours. Not recommended.

**If you don't have a GPU**, skip the training cell below and go directly to the "Loading a pre-trained adapter" section.

In [ ]:
# TRAINING CELL — Requires GPU
# Skip this cell if you don't have a GPU. The adapter will not be saved locally.

if not gpu_available:
    print("GPU not available. Skipping training.")
    print("To train, run: python training/finetune.py --persona momentum --config training/configs/momentum.yaml")
    print("Estimated time on A100: ~15 minutes")
else:
    # Import and run fine-tuning
    from training.finetune import load_config, run_finetuning
    
    config_path = project_root / "training" / "configs" / "momentum.yaml"
    config = load_config(str(config_path))
    
    # Adjust paths to be relative to project root
    config['train_data_path'] = str(project_root / config['train_data_path'])
    config['output_dir'] = str(project_root / config['output_dir'])
    
    print(f"Starting training for persona: {config['persona']}")
    print(f"Training data: {config['train_data_path']}")
    print(f"Output dir: {config['output_dir']}")
    print()
    
    run_finetuning(config)
    print("Training complete!")

## Evaluating the Fine-Tuned Adapter

After training, we want to verify that the adapter has actually learned the persona's behavioral signature. A quick qualitative check is to run inference on a few test scenarios and see if the output matches expectations.

For the momentum persona, we expect:
- BUY when prices are trending up
- SELL when prices are trending down
- HOLD when there is no clear trend
- The reasoning should explicitly reference price trends, not fair value

Below, we load either the trained adapter (if it exists) or fall back to explaining what the output would look like.

In [ ]:
# Load the fine-tuned adapter and run sample inference
# This requires the adapter to have been trained (cell above) or pre-loaded

adapter_path = project_root / "training" / "adapters" / "momentum" / "final"

if not adapter_path.exists():
    print(f"Adapter not found at: {adapter_path}")
    print("To load a pre-trained adapter, place it in training/adapters/momentum/final/")
    print("The adapter directory should contain: adapter_config.json, adapter_model.safetensors")
    print()
    print("Alternatively, the simulation uses RuleBasedAgent which does not require a trained adapter.")
    print("See simulation/agent.py for the rule-based momentum implementation.")
else:
    print(f"Found adapter at: {adapter_path}")
    
    try:
        import torch
        from peft import PeftModel
        from transformers import AutoModelForCausalLM, AutoTokenizer
        
        BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
        
        print(f"Loading base model: {BASE_MODEL}")
        tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
        base_model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL, torch_dtype=torch.float32, trust_remote_code=True
        )
        model = PeftModel.from_pretrained(base_model, str(adapter_path))
        model.eval()
        print("Model loaded.")
    except Exception as e:
        print(f"Error loading model: {e}")
        model = None
        tokenizer = None

In [ ]:
# Run sample inference if the adapter is loaded
# Shows the model's reasoning and decision for two test scenarios

test_scenarios = [
    {
        "label": "Strong uptrend",
        "price_history": [95.0, 97.5, 100.0, 103.0, 107.0],
        "news": "Volume confirms strong upward momentum.",
        "fair_value": 98.0,
        "expected_action": "BUY",
    },
    {
        "label": "Strong downtrend",
        "price_history": [110.0, 107.0, 104.0, 100.0, 96.0],
        "news": "Selling pressure continues to mount.",
        "fair_value": 105.0,
        "expected_action": "SELL",
    },
]

MOMENTUM_IDENTITY = (
    "You are a momentum trader. You believe recent price trends persist in the short run. "
    "You buy assets that have risen recently and sell assets that have fallen. "
    "You do not anchor to fundamentals."
)

for scenario in test_scenarios:
    print(f"=== Scenario: {scenario['label']} ===")
    print(f"Price history: {scenario['price_history']}")
    print(f"Expected action: {scenario['expected_action']}")
    
    if 'model' in dir() and model is not None and tokenizer is not None:
        import torch
        
        prices = scenario['price_history']
        price_change = round((prices[-1] - prices[0]) / prices[0] * 100, 2)
        
        user_content = (
            f"Price history: {prices}\nCurrent price: {prices[-1]}\n"
            f"Price change: {price_change}%\nNews: {scenario['news']}\n"
            f"Fair value: {scenario['fair_value']}\n\n"
            "Provide your trading decision as JSON: {\"reasoning\": ..., \"action\": ..., \"quantity\": ...}"
        )
        
        messages = [
            {"role": "system", "content": MOMENTUM_IDENTITY},
            {"role": "user", "content": user_content},
        ]
        
        input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(input_text, return_tensors="pt")
        
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=200, do_sample=False)
        
        new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        generated = tokenizer.decode(new_tokens, skip_special_tokens=True)
        
        print(f"Model output: {generated}")
    else:
        print("  (Model not loaded — see RuleBasedAgent in simulation/agent.py for the rule-based equivalent)")
    print()

## Summary

In this notebook we've seen:

1. **Why LoRA**: Parameter-efficient, supports multiple personas from a shared base, regularized for small datasets.

2. **Data format**: Each training example is reformatted as a chat conversation — system (persona identity) / user (market state) / assistant (reasoning + decision JSON). We train only on the assistant turn.

3. **Configuration**: r=8, alpha=16, target q_proj and v_proj. All three personas use the same config; behavioral differences emerge from the data.

4. **SFTTrainer**: Handles chat template formatting, loss masking, and PEFT integration. ~15 min on A100 per persona.

5. **Fallback**: The `RuleBasedAgent` in `simulation/agent.py` implements each persona's logic without a model, so the full simulation pipeline runs on CPU.

**Next**: Notebook `03_simulation_and_eval.ipynb` assembles the agents into the market simulation, runs the hero experiment, and checks for stylized facts.